In [4]:
import os
import tensorflow as tf
from tensorflow.keras.models import load_model

MODEL_PATH = "../models/lstm_rul_model_FD001.keras"

model = load_model(MODEL_PATH)

print("LSTM model loaded successfully.")

LSTM model loaded successfully.


In [5]:
import os
import joblib

SCALER_PATH = "../models/lstm_scaler_FD001.pkl"

print("Scaler exists:", os.path.exists(SCALER_PATH))

if os.path.exists(SCALER_PATH):
    scaler = joblib.load(SCALER_PATH)
    print("LSTM scaler loaded successfully!")
else:
    print("LSTM scaler not found. We need to save it from Day 5.")

Scaler exists: True
LSTM scaler loaded successfully!


In [6]:
import pandas as pd

columns = (
    ["unit_id", "cycle",
     "op_setting_1", "op_setting_2", "op_setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)

raw_df = pd.read_csv(
    "../data/raw/train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

print("Raw data loaded!")
print("Shape:", raw_df.shape)
raw_df.head()

Raw data loaded!
Shape: (20631, 26)


,unit_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [7]:
sensor_columns = [
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_7",
    "sensor_8",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_15",
    "sensor_17",
    "sensor_20",
    "sensor_21"
]

WINDOW_SIZE = 30

print("Number of sensors:", len(sensor_columns))
print("Window size:", WINDOW_SIZE)
print("Sensors:", sensor_columns)

Number of sensors: 12
Window size: 30
Sensors: ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


In [8]:
ENGINE_ID = 1

engine_data = raw_df[
    raw_df["unit_id"] == ENGINE_ID
].copy()

engine_data = engine_data.sort_values("cycle")

print("Engine:", ENGINE_ID)
print("Cycles:", len(engine_data))
print("First cycle:", engine_data["cycle"].min())
print("Last cycle:", engine_data["cycle"].max())

Engine: 1
Cycles: 192
First cycle: 1
Last cycle: 192


In [9]:
latest_data = engine_data[
    sensor_columns
].tail(WINDOW_SIZE)

print("Latest sensor window shape:", latest_data.shape)

Latest sensor window shape: (30, 12)


In [10]:
scaled_window = scaler.transform(latest_data)

print("Scaled window shape:", scaled_window.shape)

Scaled window shape: (30, 12)


In [11]:
import numpy as np

sequence = np.expand_dims(
    scaled_window,
    axis=0
)

print("LSTM input shape:", sequence.shape)

LSTM input shape: (1, 30, 12)
